检查模型性能，调整决策树参数（例如 max_depth, min_samples_split）

In [1]:
# 步骤1：导入必要的库
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error


In [2]:
# 步骤2：读取CSV文件
# 读取数据
data = pd.read_csv('/home/zhanyu/experiment/data-hh/my/train_86i3lia5Jj4=.csv')

In [3]:
# 步骤3：数据预处理
# 将flt_date字段转换为日期类型，并提取有用的信息（如年份、月份、日期）。
# 转换为日期类型
data['flt_date'] = pd.to_datetime(data['flt_date'])

# 提取年份、月份、日期作为新的特征
data['year'] = data['flt_date'].dt.year
data['month'] = data['flt_date'].dt.month
data['day'] = data['flt_date'].dt.day

In [4]:
# 对于字符型的特征，如a、b、c、segment、flt_no、aircraft，需要进行标签编码。
# 初始化标签编码器
le = LabelEncoder()

# 对每个需要编码的列进行编码
for col in ['a', 'b', 'c', 'segment', 'flt_no', 'aircraft']:
    data[col] = le.fit_transform(data[col])

In [5]:
# 步骤4：准备特征和目标变量
# 特征变量
X = data[['a', 'b', 'c', 'segment', 'flt_no', 'aircraft', 'duration', 'year', 'month', 'day']]

# 目标变量
y = data[['y_pax', 'c_pax']]

In [6]:
# 步骤5：拆分训练集和测试集
# 拆分数据，测试集占20%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
# 步骤6：训练决策树模型
# 由于我们有两个目标变量，可以使用多输出回归模型。
# 初始化决策树回归器
model = DecisionTreeRegressor(random_state=42,max_depth=5, min_samples_split=10)

# 训练模型
model.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=5, min_samples_split=10, random_state=42)

In [8]:
# 步骤7：模型评估
# 预测
y_pred = model.predict(X_test)

# 计算均方误差
mse_y_pax = mean_squared_error(y_test['y_pax'], y_pred[:, 0])
mse_c_pax = mean_squared_error(y_test['c_pax'], y_pred[:, 1])

print(f'y_pax的均方误差: {mse_y_pax}')
print(f'c_pax的均方误差: {mse_c_pax}')

y_pax的均方误差: 452.6502540684073
c_pax的均方误差: 3.5475855652573416


In [9]:
# 将预测结果转为 DataFrame
y_pred_df = pd.DataFrame(y_pred, columns=['y_pax_pred', 'c_pax_pred'])

# 将实际值和预测值合并
results = X_test.copy()  # 复制测试集特征
results['y_pax_actual'] = y_test['y_pax'].values
results['c_pax_actual'] = y_test['c_pax'].values
results['y_pax_pred'] = y_pred_df['y_pax_pred']
results['c_pax_pred'] = y_pred_df['c_pax_pred']

# 显示部分预测结果
print("部分预测结果：")
print(results.head(10))

# 保存结果为CSV文件
save_path = '/home/zhanyu/experiment/data-hh/my/prediction_results_jcs.csv'
results.to_csv(save_path, index=False)

print(f"预测结果已保存至: {save_path}")


部分预测结果：
     a  b  c  segment  flt_no  aircraft  duration  year  month  day  \
334  0  0  0        1       0         0      2.30  2023     11   25   
137  0  0  0        2       0         0      4.31  2023      2   15   
72   0  0  0        0       0         0      2.05  2023      1   25   
365  0  0  0        2       0         0      4.43  2023     12    5   
73   0  0  0        1       0         0      2.40  2023      1   25   
294  0  0  0        1       0         0      2.42  2023     11   12   
418  0  0  0        0       0         1      1.82  2023     12   23   
220  0  0  0        2       0         0      4.50  2023      3   15   
140  0  0  0        0       0         0      1.85  2023      2   16   
425  0  0  0        0       0         0      1.88  2023     12   25   

     y_pax_actual  c_pax_actual  y_pax_pred  c_pax_pred  
334            74             0         NaN         NaN  
137            73             5         NaN         NaN  
72             89             0   71

4o

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

# Step 1: Read the CSV file
data = pd.read_csv('/home/zhanyu/experiment/data-hh/my/train_86i3lia5Jj4=.csv')

# Step 2: Encode categorical variables
categorical_columns = ['flt_date', 'a', 'b', 'c', 'segment', 'flt_no', 'aircraft']
label_encoders = {col: LabelEncoder() for col in categorical_columns}

for col in categorical_columns:
    data[col] = label_encoders[col].fit_transform(data[col])

# Step 3: Split features and targets
features = data[['flt_date', 'a', 'b', 'c', 'segment', 'flt_no', 'aircraft', 'duration']]
target_y_pax = data['y_pax']
target_c_pax = data['c_pax']

# Step 4: Train-test split
X_train, X_test, y_pax_train, y_pax_test = train_test_split(features, target_y_pax, test_size=0.2, random_state=42)
_, _, c_pax_train, c_pax_test = train_test_split(features, target_c_pax, test_size=0.2, random_state=42)

# Step 5: Train Decision Tree for y_pax
model_y_pax = DecisionTreeRegressor(random_state=42,max_depth=5, min_samples_split=10)
model_y_pax.fit(X_train, y_pax_train)

# Step 6: Train Decision Tree for c_pax
model_c_pax = DecisionTreeRegressor(random_state=42,max_depth=5, min_samples_split=10)
model_c_pax.fit(X_train, c_pax_train)

# Step 7: Make predictions and evaluate
y_pax_pred = model_y_pax.predict(X_test)
c_pax_pred = model_c_pax.predict(X_test)

# Evaluate y_pax model
print("y_pax Model Performance:")
print("MSE:", mean_squared_error(y_pax_test, y_pax_pred))
print("R²:", r2_score(y_pax_test, y_pax_pred))

# Evaluate c_pax model
print("c_pax Model Performance:")
print("MSE:", mean_squared_error(c_pax_test, c_pax_pred))
print("R²:", r2_score(c_pax_test, c_pax_pred))


y_pax Model Performance:
MSE: 385.3156700266306
R²: 0.39080593355161364
c_pax Model Performance:
MSE: 3.831060710119503
R²: -0.2552048264748752


In [11]:
# Create a DataFrame to display predictions vs actuals for y_pax
results_y_pax = pd.DataFrame({
    'Actual_y_pax': y_pax_test,
    'Predicted_y_pax': y_pax_pred
}).reset_index(drop=True)

# Create a DataFrame to display predictions vs actuals for c_pax
results_c_pax = pd.DataFrame({
    'Actual_c_pax': c_pax_test,
    'Predicted_c_pax': c_pax_pred
}).reset_index(drop=True)

# Display first few rows of each result
print("Sample y_pax Predictions:")
print(results_y_pax.head())

print("\nSample c_pax Predictions:")
print(results_c_pax.head())

Sample y_pax Predictions:
   Actual_y_pax  Predicted_y_pax
0            74        74.539216
1            73        70.500000
2            89        93.833333
3            28        54.900000
4            54        74.539216

Sample c_pax Predictions:
   Actual_c_pax  Predicted_c_pax
0             0         1.142857
1             5         2.071429
2             0         3.764706
3             2         2.071429
4             0         1.323529
